In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ml = importlib.import_module("flows.05_ml_models")
sns.set_theme(style="whitegrid", context="notebook")
X, y = ml.load_dataset()
print(X.shape, y.mean())

In [ ]:
X_train, X_test, y_train, y_test = ml.split(X, y)
print(len(X_train), len(X_test), y_train.mean(), y_test.mean())

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    cross_validate,
)

cv = StratifiedKFold(n_splits=ml.settings.cv_folds, shuffle=True, random_state=ml.settings.random_state)

def metricas(y_real, pred):
    return {'exactitud': accuracy_score(y_real, pred),
            'precision': precision_score(y_real, pred, zero_division=0),
            'recall': recall_score(y_real, pred),
            'f1': f1_score(y_real, pred)}

In [ ]:
pred_ingenuo = cross_val_predict(DummyClassifier(strategy='most_frequent'), X_train, y_train, cv=cv)
pred_regla = (X_train['Sex'] == 'female').astype(int)
pd.DataFrame({'ingenuo': metricas(y_train, pred_ingenuo),
              'regla_genero': metricas(y_train, pred_regla)}).round(3)

In [ ]:
edad_todos = X.loc[X['Sex'] == 'female', 'Age'].mean()
edad_train = X_train.loc[X_train['Sex'] == 'female', 'Age'].mean()
edad_todos, edad_train, edad_todos - edad_train

In [ ]:
pipe = ml.build_pipeline(DummyClassifier())
pre = pipe[:-1].fit(X_train)
Z = pre.transform(X_test)
print(np.isnan(Z).any(), Z.shape)
list(pre['prep'].get_feature_names_out())

In [ ]:
resultados = ml.cross_validate_candidates(X_train, y_train)
resultados.round(3)

In [ ]:
filas = []
for nombre, modelo in ml.get_candidates(ml.settings.random_state).items():
    res = cross_validate(ml.build_pipeline(modelo), X_train, y_train, cv=cv, scoring='roc_auc')
    for puntaje in res['test_score']:
        filas.append({'modelo': nombre, 'roc_auc': puntaje})
folds = pd.DataFrame(filas)
sns.boxplot(data=folds, x='modelo', y='roc_auc')
plt.title('ROC-AUC por pliegue'); plt.show()

In [ ]:
resultados[['model', 'gap_train_val']].round(3)

In [ ]:
finalista = 'hgb'   
modelo = ml.get_candidates(ml.settings.random_state)[finalista]
pipe_finalista = ml.build_pipeline(modelo)

from sklearn.metrics import ConfusionMatrixDisplay, classification_report

proba = cross_val_predict(pipe_finalista, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
pred = (proba >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_train, pred)
plt.show()
print(classification_report(y_train, pred))

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, ax = plt.subplots()
for nombre in ['logreg', 'forest', 'hgb']:   # tus finalistas
    p = cross_val_predict(ml.build_pipeline(ml.get_candidates(ml.settings.random_state)[nombre]),
                          X_train, y_train, cv=cv, method='predict_proba')[:, 1]
    RocCurveDisplay.from_predictions(y_train, p, name=nombre, ax=ax)
plt.show()

In [ ]:
PrecisionRecallDisplay.from_predictions(y_train, proba)
plt.show()

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, precision_recall_curve

prob_true, prob_pred = calibration_curve(y_train, proba, n_bins=8)
plt.plot(prob_pred, prob_true, marker='o', label='modelo')
plt.plot([0, 1], [0, 1], '--', label='calibración perfecta')
plt.xlabel('Probabilidad predicha'); plt.ylabel('Fracción real de sobrevivientes'); plt.legend()
plt.show()

In [ ]:
print('Brier:', brier_score_loss(y_train, proba))
prec, rec, umbrales = precision_recall_curve(y_train, proba)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
print('Umbral que maximiza F1:', umbrales[f1s.argmax()], 'F1:', f1s.max())

In [ ]:
analisis = X_train.copy()
analisis['y'] = y_train
analisis['pred'] = pred
analisis['proba'] = proba
analisis['error'] = analisis['pred'] != analisis['y']

def por_grupo(col):
    g = analisis.groupby(col, observed=True)
    return pd.DataFrame({'n': g.size(), 'exactitud': 1 - g['error'].mean(),
                         'recall': analisis[analisis['y'] == 1].groupby(col, observed=True)['pred'].mean()})

analisis['edad_grupo'] = pd.cut(analisis['Age'], [0, 16, 40, 80])
display(por_grupo('Sex'))
display(por_grupo('Pclass'))
display(por_grupo('edad_grupo'))

In [ ]:
analisis['confianza_error'] = np.where(analisis['error'], (analisis['proba'] - 0.5).abs(), 0)
analisis.sort_values('confianza_error', ascending=False).head(10)[['Sex', 'Pclass', 'Age', 'Fare', 'y', 'proba']]

In [ ]:
from sklearn.model_selection import learning_curve

tam, tr, va = learning_curve(pipe_finalista, X_train, y_train, cv=cv,
                             train_sizes=np.linspace(0.2, 1.0, 5), scoring='roc_auc')
plt.plot(tam, tr.mean(axis=1), label='entrenamiento')
plt.fill_between(tam, tr.mean(axis=1) - tr.std(axis=1), tr.mean(axis=1) + tr.std(axis=1), alpha=0.2)
plt.plot(tam, va.mean(axis=1), label='validación')
plt.fill_between(tam, va.mean(axis=1) - va.std(axis=1), va.mean(axis=1) + va.std(axis=1), alpha=0.2)
plt.xlabel('Pasajeros de entrenamiento'); plt.ylabel('ROC-AUC'); plt.legend()
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = ml.get_param_grids()[finalista]
gs = GridSearchCV(ml.build_pipeline(ml.get_candidates(ml.settings.random_state)[finalista]), grid,
                  scoring=ml.PRIMARY_METRIC, cv=cv, n_jobs=-1, refit=True)
gs.fit(X_train, y_train)
print(gs.best_params_, gs.best_score_)
pd.DataFrame(gs.cv_results_).sort_values('rank_test_score')[
    ['rank_test_score', 'mean_test_score', 'std_test_score']].head(5)

In [ ]:
base = cross_val_score(ml.build_pipeline(ml.get_candidates(ml.settings.random_state)[finalista]),
                       X_train, y_train, cv=cv, scoring='roc_auc')
ajustado = cross_val_score(gs.best_estimator_, X_train, y_train, cv=cv, scoring='roc_auc')
dif = ajustado - base
dif.mean(), dif.std()

In [ ]:
pipe_lr = ml.build_pipeline(ml.get_candidates(ml.settings.random_state)['logreg']).fit(X_train, y_train)
nombres = pipe_lr['prep'].get_feature_names_out()
coefs = pipe_lr['model'].coef_[0]
tabla = pd.DataFrame({'variable': nombres, 'coef': coefs, 'odds_ratio': np.exp(coefs)})
tabla.reindex(tabla['coef'].abs().sort_values(ascending=False).index)

In [ ]:
from sklearn.inspection import permutation_importance

X_tr, X_val, y_tr, y_val = ml.split(X_train, y_train, test_size=0.25)
pipe_perm = ml.build_pipeline(ml.get_candidates(ml.settings.random_state)[finalista]).fit(X_tr, y_tr)
r = permutation_importance(pipe_perm, X_val, y_val, scoring='roc_auc', n_repeats=30, random_state=42)
imp = pd.DataFrame({'variable': X_val.columns, 'media': r.importances_mean,
                    'std': r.importances_std}).sort_values('media')
plt.barh(imp['variable'], imp['media'], xerr=imp['std'])
plt.xlabel('Caída del ROC-AUC al barajar la variable')
plt.show()

In [ ]:
configuraciones = {
    'A base': (ml.NUM_FEATURES, ml.CAT_FEATURES),
    'B base + cabina': (ml.NUM_FEATURES + ml.CABIN_NUM, ml.CAT_FEATURES + ml.CABIN_CAT),
    'C sin TTitulo': (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != 'TTitulo']),
    'D sin Sex': (ml.NUM_FEATURES, [c for c in ml.CAT_FEATURES if c != 'Sex']),
}
filas = []
for nombre, (num, cat) in configuraciones.items():
    pipe_a = ml.build_pipeline(ml.get_candidates(ml.settings.random_state)[finalista], num, cat)
    res = cross_val_score(pipe_a, X_train, y_train, cv=cv, scoring='roc_auc')
    filas.append({'configuracion': nombre, 'auc_media': res.mean(), 'auc_std': res.std()})
ablacion = pd.DataFrame(filas)
ablacion['delta_vs_A'] = ablacion['auc_media'] - ablacion.loc[0, 'auc_media']
ablacion.round(4)

In [ ]:
pipe_eval = ml.build_pipeline(ml.get_candidates(ml.settings.random_state)[finalista]
                              ).set_params(**gs.best_params_).fit(X_train, y_train)
resultado = ml.evaluate_on_test(pipe_eval, X_test, y_test, threshold=ml.THRESHOLD)
proba_test = pipe_eval.predict_proba(X_test)[:, 1]
print(resultado)
print('IC95% ROC-AUC (bootstrap):', ml.bootstrap_ci(y_test, proba_test))

In [ ]:
pred_regla_test = (X_test['Sex'] == 'female').astype(int)
pred_ingenuo_test = DummyClassifier(strategy='most_frequent').fit(X_train, y_train).predict(X_test)
pd.DataFrame({'regla_genero': metricas(y_test, pred_regla_test),
              'ingenuo': metricas(y_test, pred_ingenuo_test),
              'modelo': metricas(y_test, (proba_test >= ml.THRESHOLD).astype(int))}).round(3)

In [ ]:
from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME, MODELS

de = importlib.import_module('flows.01_data_engineering')

pipe_guardado, meta = ml.load_model(MODELS / 'titanic_model_v1.joblib')
nuevos = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
nuevos = (nuevos.pipe(de.add_person).pipe(de.add_cabin_features).pipe(de.add_family_features)
          .pipe(de.add_title).pipe(de.consolidate_titles).pipe(de.impute_embarked).pipe(de.impute_age))
X_nuevos = nuevos[ml.CANDIDATE_COLUMNS].copy()
X_nuevos['Age'] = nuevos['AgeOriginal']

salida = ml.predict(pipe_guardado, X_nuevos)
print(salida.isna().sum().sum())
salida['proba_survive'].describe().round(3)

In [ ]:
sorted(set(X_nuevos['TTitulo']) - set(X['TTitulo']))